In [1]:
import openeo
import os
import geopandas as gpd
import rasterio
import glob
from rasterio.mask import mask
from rasterio.merge import merge
from pathlib import Path
import numpy as np
from openeo.processes import quantiles, array_element, if_

import time

In [2]:
years = [2016, 2017, 2025, 2026]
# 
base_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/")

### 6) Calculate indices & apply threshold

In [3]:
indices_out = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/ndvi")

for year in years:
    in_folder = base_folder / str(year)
    out_folder = indices_out / str(year)
    out_folder.mkdir(parents=True, exist_ok=True)  

    print("Looking in:", in_folder)
    print("Folder exists:", in_folder.exists())

    for f in sorted(in_folder.glob("S2_mosaic*.tiff")):
        with rasterio.open(f) as src:
            print(f"File: {f.name}, CRS: {src.crs}, Size: {src.width}x{src.height}")

            red   = src.read(3).astype(np.float32)
            nir   = src.read(4).astype(np.float32)
            nodata = src.nodata
            profile = src.profile.copy()

        # Valid-pixel mask
        valid = (red >= 0) & (nir >= 0) 
        if nodata is not None:
            valid &= (red != nodata) & (nir != nodata)

        with np.errstate(divide="ignore", invalid="ignore"):
            ndvi = np.where(valid, (nir - red) / (nir + red), np.nan)


        profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)

        outputs = {
            "ndvi": ndvi,
        }

        for name, arr in outputs.items():
            with rasterio.open(out_folder / f"{f.stem}_{name}.tiff", "w", **profile) as dst:
                dst.write(arr.astype(np.float32), 1)

Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/2016
Folder exists: True
File: S2_mosaic_east_2016.tiff, CRS: EPSG:32632, Size: 4294x2378


/tmp/ipykernel_2332323/451268768.py:15: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  red   = src.read(3).astype(np.float32)
/tmp/ipykernel_2332323/451268768.py:16: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  nir   = src.read(4).astype(np.float32)


File: S2_mosaic_west_2016.tiff, CRS: EPSG:32632, Size: 6370x6256
Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/2017
Folder exists: True
File: S2_mosaic_east_2017.tiff, CRS: EPSG:32632, Size: 4294x2378
File: S2_mosaic_west_2017.tiff, CRS: EPSG:32632, Size: 6370x6256
Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/2025
Folder exists: True
File: S2_mosaic_east_2025.tiff, CRS: EPSG:32632, Size: 4294x2378
File: S2_mosaic_west_2025.tiff, CRS: EPSG:32632, Size: 6370x6256
Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/2026
Folder exists: True
File: S2_mosaic_east_2026.tiff, CRS: EPSG:32632, Size: 4294x2378
File: S2_mosaic_west_2026.tiff, CRS: EPSG:32632, Size: 6370x6256
